# Cleaning Data Mentah Pengaduan Surabaya 




In [ ]:
from pathlib import Path
import re, json, unicodedata, warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

BASE_DIR = Path.cwd()
SEARCH_DIRS = [BASE_DIR, BASE_DIR / "project_raw", BASE_DIR.parent, BASE_DIR.parent / "project_raw"]

TAHUN_LIST  = [2023, 2024, 2025]
N_KELURAHAN = 153
BBOX_LAT    = (-7.45, -7.10)     # batas pemeriksaan Surabaya
BBOX_LON    = (112.55, 112.90)

KECAMATAN_RESMI = [
    "Asemrowo", "Benowo", "Bubutan", "Bulak", "Dukuh Pakis", "Gayungan", "Genteng", "Gubeng",
    "Gunung Anyar", "Jambangan", "Karang Pilang", "Kenjeran", "Krembangan", "Lakarsantri",
    "Mulyorejo", "Pabean Cantian", "Pakal", "Rungkut", "Sambikerep", "Sawahan", "Semampir",
    "Simokerto", "Sukolilo", "Sukomanunggal", "Tambaksari", "Tandes", "Tegalsari",
    "Tenggilis Mejoyo", "Wiyung", "Wonocolo", "Wonokromo",
]

def find_file(patterns, deskripsi):
    dicari = []
    for d in SEARCH_DIRS:
        for pat in patterns:
            hits = sorted(p for p in d.glob(pat) if not p.name.startswith("~$"))
            if hits:
                return hits[0]
        dicari.append(str(d))
    raise FileNotFoundError(f"File {deskripsi} tidak ditemukan. Folder yang dicari: {dicari}. "
                            f"Pola: {patterns}")

FILES = {
    y: find_file([f"mediacenter.keluhan (Tahun {y}).xlsx", f"*keluhan*{y}*.xlsx"], f"pengaduan {y}")
    for y in TAHUN_LIST
}
COORD_FILE = find_file(
    ["koordinat_kelurahan_2023_2025.csv", "koordinat_kelurahan*2023_2025*.csv", "koordinat_kelurahan*.csv"],
    "koordinat kelurahan",
)


def normalize_location(value):
    if pd.isna(value):
        return pd.NA
    v = unicodedata.normalize("NFKC", str(value)).casefold()
    v = re.sub(r"\s+", " ", v).strip()
    return v if v else pd.NA

def kec_key(value):
    # kunci kecamatan tanpa spasi: 'Gunung Anyar' == 'Gununganyar'
    n = normalize_location(value)
    return pd.NA if pd.isna(n) else n.replace(" ", "")

KEC_RESMI_KEY = {kec_key(k) for k in KECAMATAN_RESMI}
PHONE_RE = re.compile(r"(?<!\d)(?:\+?62|0)[\s.\-]?8\d(?:[\s.\-]?\d){7,10}(?!\d)")

print("Folder kerja :", BASE_DIR)
for y, p in FILES.items():
    print(f"Pengaduan {y}:", p)
print("Koordinat    :", COORD_FILE)

In [ ]:
OUTPUT_DIR = BASE_DIR / "cleaning_output"; OUTPUT_DIR.mkdir(exist_ok=True)


KELURAHAN_ALIAS = {}

LOKASI_MANUAL = {}

MASK_TELEPON  = True                
TOKEN_TELEPON = "[NOMOR_TELEPON]"
MASK_PII_LAIN = True                
TOKEN_ID, TOKEN_EMAIL, TOKEN_AKUN = "[NOMOR_IDENTITAS]", "[EMAIL]", "[AKUN]"
MASK_URL  = True                   
TOKEN_URL = "[URL]"

ALLOW_FALLBACK_COORDINATE = False  
SHORT_THRESHOLD = 10                
LONG_THRESHOLD  = 2000             
MIN_N_WARN      = 5                


TOPIK_OVERRIDE = {
    "Parkir Mobil di Jalan Lingkungan/Kawasan Permukiman": "Perhubungan & Lalu Lintas",       
    "Pembersihan Sampah Di Saluran/Sungai, Pedestrian, dan Jalan": "Kebersihan & Lingkungan",  
    "Marka Jalan": "Perhubungan & Lalu Lintas",            
    "Penutupan Jalan Utama": "Perhubungan & Lalu Lintas", 
    "Pembatas jalan": "Perhubungan & Lalu Lintas",         
    "CCTV Jalan": "Perhubungan & Lalu Lintas",            
    "Kelas Jalan": "Perhubungan & Lalu Lintas",            #
    "Pembersihan Sungai Akibat Sedimen Saluran": "Banjir & Drainase",   
    "Normalisasi Saluran Permukiman": "Banjir & Drainase",             
    "Perbaikan Tanggul / Plengsengan": "Banjir & Drainase",          
    "Pavingisasi": "Infrastruktur Jalan & Saluran",                 
    "Miras (Minuman Keras)": "Keamanan & Ketertiban",      
    "CCTV Lingkungan": "Keamanan & Ketertiban",           
    "Beasiswa SMA (Pemuda Tangguh)": "Pendidikan",             
    "Beasiswa Mahasiswa": "Pendidikan",
    "Beasiswa Pemuda Tangguh": "Pendidikan",
    "Beasiswa Bibit Unggul": "Pendidikan",
    "Pengerukan Saluran Permukiman": "Banjir & Drainase",     
    "Saluran Air Di Kampung/Perumahan": "Banjir & Drainase",   
    "Keamanan Website/Aplikasi Pemerintah Kota Surabaya": "Lainnya",  
    "Pajak Penerangan Jalan": "Pajak & Retribusi Daerah",  
}


## 1. Load data raw
`Tahun_file` = tahun berdasarkan file sumber (metadata). Tahun analisis (`tahun`) diturunkan dari tanggal di bagian 3.

In [ ]:
frames = []
for y, p in FILES.items():
    d = pd.read_excel(p)
    d["Tahun_file"] = y
    frames.append(d)
df_raw = pd.concat(frames, ignore_index=True)
df_raw["raw_row"] = np.arange(len(df_raw))           
print("Shape gabungan:", df_raw.shape)

def _clean_col(c):
    return re.sub(r"\s+", " ", str(c)).strip()

df = df_raw.copy()
df.columns = [_clean_col(c) for c in df.columns]

WAJIB = ["No Ticket", "Instansi", "Topik", "Kategori", "Channel", "Keluhan", "Kelurahan", "Kecamatan", "Tanggal Keluhan"]
hilang = [c for c in WAJIB if c not in df.columns]
assert not hilang, f"Kolom wajib tidak ada: {hilang}"


for col in ["Instansi", "Topik", "Kategori", "Channel", "Keluhan", "Kelurahan", "Kecamatan"]:
    df[col] = df[col].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    df[col] = df[col].mask(df[col] == "", pd.NA)
print("Baris mentah:", len(df))

## 2. Tiket & duplikasi

In [ ]:
n_dup_baris = int(df.drop(columns=["raw_row"]).duplicated().sum())
n_dup_tiket_dalam = int(df.duplicated(["Tahun_file", "No Ticket"]).sum())
n_dup_tiket_total = int(df["No Ticket"].duplicated().sum())
print("Duplikat baris utuh          :", n_dup_baris)
print("Duplikat tiket dalam 1 tahun :", n_dup_tiket_dalam)
print("Duplikat tiket antar tahun   :", n_dup_tiket_total - n_dup_tiket_dalam)


assert n_dup_tiket_total == 0, "Ada duplikat tiket: tentukan aturan dedup secara eksplisit sebelum lanjut."
print("OK: No Ticket unik pada seluruh data.")

## 3. Tanggal & tahun analisis
`tahun` diturunkan dari `Tanggal Keluhan` dan **harus** sama dengan tahun file.

In [ ]:
df["tanggal_dt"] = pd.to_datetime(df["Tanggal Keluhan"], errors="coerce")
df["tahun"] = df["tanggal_dt"].dt.year

n_tgl_invalid = int(df["tanggal_dt"].isna().sum())
n_tahun_beda  = int((df["tahun"].notna() & (df["tahun"] != df["Tahun_file"])).sum())
print("Tanggal invalid                       :", n_tgl_invalid)
print("Tahun tanggal != tahun file           :", n_tahun_beda)
assert n_tahun_beda == 0, "Tahun pada tanggal tidak sama dengan tahun file: periksa sebelum lanjut."
display(df.groupby("Tahun_file")["tanggal_dt"].agg(["min", "max", "count"]))


tgl_invalid = df.loc[df["tanggal_dt"].isna(), ["raw_row", "No Ticket", "Tahun_file", "Tanggal Keluhan", "Kelurahan", "Kecamatan"]]
print(f"\nBaris dengan tanggal invalid ({len(tgl_invalid)}) - tinjau sebelum lanjut:")
display(tgl_invalid.head(20))
tgl_invalid.to_csv(OUTPUT_DIR / "tanggal_invalid_audit.csv", index=False, encoding="utf-8-sig")


## 4. Master koordinat kelurahan
File koordinat adalah hasil tahap wrangling sebelumnya, jadi kualitasnya divalidasi di sini — bukan diasumsikan benar.

In [ ]:
coord = pd.read_csv(COORD_FILE)
need = ["kelurahan", "kecamatan", "lat_kelurahan", "lon_kelurahan"]
assert all(c in coord.columns for c in need), f"Kolom koordinat wajib: {need}"

master = coord.copy().reset_index(drop=True)
master["master_id"] = master.index
master["kelurahan"] = master["kelurahan"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
master["kecamatan"] = master["kecamatan"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
master["kel_norm"]  = master["kelurahan"].map(normalize_location)
master["kec_k"]     = master["kecamatan"].map(kec_key)
master["kec_norm"]  = master["kecamatan"].map(normalize_location)

def _kel_base(kel_norm, kec_norm):
    if pd.isna(kel_norm) or pd.isna(kec_norm):
        return kel_norm
    suf = " " + kec_norm
    return kel_norm[:-len(suf)].strip() if kel_norm.endswith(suf) and len(kel_norm) > len(suf) else kel_norm

master["kel_base"]   = [_kel_base(a, b) for a, b in zip(master["kel_norm"], master["kec_norm"])]
master["lokasi_key"] = master["kel_base"].astype(str) + "|" + master["kec_k"].astype(str)  
master["lat"] = pd.to_numeric(master["lat_kelurahan"], errors="coerce")
master["lon"] = pd.to_numeric(master["lon_kelurahan"], errors="coerce")
if "sumber_koordinat" not in master.columns:
    print("PERINGATAN: kolom sumber_koordinat tidak ada; asal koordinat tidak dapat diaudit.")
    master["sumber_koordinat"] = "tidak_diketahui"

cek_master = {
    "jumlah_baris": len(master),
    "nama_kelurahan_duplikat": int(master["kel_norm"].duplicated().sum()),
    "identitas_kel_kec_duplikat": int(master["lokasi_key"].duplicated().sum()),
    "label_tanpa_spasi_duplikat": int(master["kel_norm"].str.replace(" ", "", regex=False).duplicated().sum()),
    "koordinat_kosong": int(master[["lat", "lon"]].isna().any(axis=1).sum()),
    "di_luar_bbox": int((~master["lat"].between(*BBOX_LAT) | ~master["lon"].between(*BBOX_LON)).sum()),
    "lat_lon_duplikat": int(master.duplicated(["lat", "lon"]).sum()),
    "kecamatan_non_resmi": int((~master["kec_k"].isin(KEC_RESMI_KEY)).sum()),
    "baris_fallback": int(master["sumber_koordinat"].astype(str).str.contains("fallback", case=False).sum()),
}
display(pd.Series(cek_master, name="nilai").to_frame())
print("sumber_koordinat:"); display(master["sumber_koordinat"].value_counts(dropna=False).rename("n").to_frame())

assert cek_master["jumlah_baris"] == N_KELURAHAN, f"Master harus {N_KELURAHAN} baris."
assert cek_master["identitas_kel_kec_duplikat"] == 0, "Pasangan kelurahan+kecamatan di master tidak unik -> identitas administratif ganda."
assert cek_master["nama_kelurahan_duplikat"] == 0, "Label kelurahan di master tidak unik: bedakan labelnya (mis. tambahkan nama kecamatan) agar tiap kelurahan punya label sendiri."
assert cek_master["label_tanpa_spasi_duplikat"] == 0, "Ada dua label master yang sama bila spasi dihapus."
assert cek_master["koordinat_kosong"] == 0 and cek_master["di_luar_bbox"] == 0, "Koordinat master tidak valid."
assert cek_master["lat_lon_duplikat"] == 0, "Ada koordinat sama dipakai >1 kelurahan."
assert cek_master["kecamatan_non_resmi"] == 0, "Ada kecamatan non-resmi di master."
if not ALLOW_FALLBACK_COORDINATE:
    assert cek_master["baris_fallback"] == 0, "Ada koordinat fallback kecamatan di master (bukan koordinat kelurahan)."
print("OK: master koordinat valid.")

## 5. Pencocokan lokasi



In [ ]:
df["kel_norm"] = df["Kelurahan"].map(normalize_location)
df["kel_norm"] = df["kel_norm"].map(lambda x: x if pd.isna(x) else KELURAHAN_ALIAS.get(x, x))
df["kec_k_raw"] = df["Kecamatan"].map(kec_key)
df["_kec_key"] = df["kec_k_raw"].fillna("").astype(str)      
LABEL_MAP         = dict(zip(master["kel_norm"], master["master_id"]))
LABEL_KEC_MAP     = {(a, b): i for i, a, b in zip(master["master_id"], master["kel_norm"], master["kec_k"])}
LABEL_TANPA_SPASI = {a.replace(" ", ""): i for i, a in zip(master["master_id"], master["kel_norm"])}
KANDIDAT = {}                                               
for i, lab, base in zip(master["master_id"], master["kel_norm"], master["kel_base"]):
    KANDIDAT.setdefault(lab, set()).add(i)
    KANDIDAT.setdefault(base, set()).add(i)
KELUARGA_NAMA = {b: ids for b, ids in master.groupby("kel_base")["master_id"].apply(list).items() if len(ids) > 1}   
KEC_MASTER  = dict(zip(master["master_id"], master["kec_k"]))
BASE_MASTER = dict(zip(master["master_id"], master["kel_base"]))
for (_k, _c), _lab in LOKASI_MANUAL.items():
    assert _lab in LABEL_MAP, f"LOKASI_MANUAL menunjuk label yang tidak ada di master: {_lab}"

def _resolve(kel, keck):
    if (kel, keck) in LOKASI_MANUAL:
        return LABEL_MAP[LOKASI_MANUAL[(kel, keck)]], "MANUAL_TERVERIFIKASI"
    if (kel, keck) in LABEL_KEC_MAP:
        return LABEL_KEC_MAP[(kel, keck)], "PASANGAN_EXACT"
    if keck and (kel.replace(" ", "") + keck) in LABEL_TANPA_SPASI:
        return LABEL_TANPA_SPASI[kel.replace(" ", "") + keck], "LABEL_MASTER"
    ids = KANDIDAT.get(kel, set())
    if len(ids) == 1:
        i0 = next(iter(ids))
        kembar = KELUARGA_NAMA.get(BASE_MASTER[i0])
        if kembar and keck:                      
            cocok = [j for j in kembar if KEC_MASTER[j] == keck]
            if len(cocok) == 1:
                return cocok[0], "KECAMATAN_PEMBEDA"  
            return None, "AMBIGU"                  
        return i0, "NAMA_UNIK"                   
    if len(ids) > 1:
        return None, "AMBIGU"                     
    return None, "TIDAK_ADA"

pairs = df.loc[df["kel_norm"].notna(), ["kel_norm", "_kec_key"]].drop_duplicates().reset_index(drop=True)
_hasil = [_resolve(a, b) for a, b in zip(pairs["kel_norm"], pairs["_kec_key"])]
pairs["master_id"] = pd.array([h[0] for h in _hasil], dtype="Int64")
pairs["metode_cocok"] = [h[1] for h in _hasil]

df = df.drop(columns=[c for c in ["master_id", "metode_cocok", "kelurahan_master", "kecamatan_master", "kec_k_master",
                                  "lat", "lon", "sumber_koordinat", "lokasi_key", "kode_kel"] if c in df.columns])
df = df.merge(pairs, on=["kel_norm", "_kec_key"], how="left", validate="many_to_one")
df["metode_cocok"] = df["metode_cocok"].fillna("MISSING_LOCATION")

m_cols = ["master_id", "kelurahan", "kecamatan", "kec_k", "lat", "lon", "sumber_koordinat", "lokasi_key"]
if "kode_kel" in master.columns:                            
    m_cols.append("kode_kel")
m = master[m_cols].rename(columns={"kelurahan": "kelurahan_master", "kecamatan": "kecamatan_master", "kec_k": "kec_k_master"})
m["master_id"] = m["master_id"].astype("Int64")
df = df.merge(m, on="master_id", how="left", validate="many_to_one")

df["coordinate_status"] = np.select(
    [df["kel_norm"].isna(),
     df["metode_cocok"] == "AMBIGU",
     df["lat"].isna() | df["lon"].isna(),
     ~df["lat"].between(*BBOX_LAT) | ~df["lon"].between(*BBOX_LON)],
    ["MISSING_LOCATION", "AMBIGUOUS_LOCATION", "NO_COORDINATE_MATCH", "INVALID_COORDINATE"],
    default="MATCH_COORDINATE",
)
display(pd.crosstab(df["Tahun_file"], df["coordinate_status"], margins=True))
print("Metode pencocokan:")
display(df["metode_cocok"].value_counts().rename("n_baris").to_frame())

tak_cocok = (df.loc[df["coordinate_status"] == "NO_COORDINATE_MATCH", ["Kelurahan", "Kecamatan"]]
             .value_counts(dropna=False).reset_index(name="n_baris"))
print("Nama kelurahan di pengaduan yang TIDAK ada di master koordinat:")
display(tak_cocok)
if len(tak_cocok):
    print("-> Perbaiki lewat KELURAHAN_ALIAS di sel konfigurasi (mis. beda ejaan), lalu jalankan ulang.")
tak_cocok.to_csv(OUTPUT_DIR / "lokasi_tidak_cocok_audit.csv", index=False, encoding="utf-8-sig")

ambigu = (df.loc[df["coordinate_status"] == "AMBIGUOUS_LOCATION", ["Kelurahan", "Kecamatan"]]
          .value_counts(dropna=False).reset_index(name="n_baris"))
print("Nama kelurahan AMBIGU (lebih dari satu kandidat di master; kecamatan di data tidak membedakan):")
display(ambigu)
if len(ambigu):
    print("-> Jangan dikoreksi dari nama saja. Verifikasi ke sumber administratif, lalu isi LOKASI_MANUAL, dan jalankan ulang.")
ambigu.to_csv(OUTPUT_DIR / "lokasi_ambigu_audit.csv", index=False, encoding="utf-8-sig")


In [ ]:
ada = df["master_id"].notna()
beda = df["kec_k_raw"].fillna("").astype(str) != df["kec_k_master"].fillna("").astype(str)
df["kecamatan_dikoreksi"] = (ada & beda).astype(bool)

audit_kec = (df.loc[df["kecamatan_dikoreksi"], ["Kelurahan", "Kecamatan", "kelurahan_master", "kecamatan_master", "metode_cocok"]]
             .value_counts(dropna=False).reset_index(name="n_baris")
             .rename(columns={"Kecamatan": "kecamatan_di_data", "kecamatan_master": "kecamatan_dikoreksi_ke"}))
print("Baris dengan Kecamatan salah-input yang dikoreksi mengikuti master:", int(df["kecamatan_dikoreksi"].sum()))
display(audit_kec)
audit_kec.to_csv(OUTPUT_DIR / "kecamatan_correction_audit.csv", index=False, encoding="utf-8-sig")


df["kelurahan_dikoreksi"] = (df["metode_cocok"] == "KECAMATAN_PEMBEDA").astype(bool)
audit_kel = df.loc[df["kelurahan_dikoreksi"], ["No Ticket", "Tanggal Keluhan", "Kelurahan", "Kecamatan", "kelurahan_master", "kecamatan_master", "Topik", "Keluhan"]].copy()
audit_kel["Keluhan"] = audit_kel["Keluhan"].astype(str).str.replace(r"\s+", " ", regex=True).str.slice(0, 160)
audit_kel = audit_kel.rename(columns={"Kelurahan": "kelurahan_di_data", "Kecamatan": "kecamatan_di_data", "kelurahan_master": "kelurahan_dikoreksi_ke"})
print("Baris dengan LABEL kelurahan (nama kembar) dikoreksi mengikuti kecamatan mentah:", int(df["kelurahan_dikoreksi"].sum()))
display(audit_kel.drop(columns=["Keluhan", "Tanggal Keluhan"]).groupby(["kelurahan_di_data", "kecamatan_di_data", "kelurahan_dikoreksi_ke"]).size().rename("n_baris").reset_index())
audit_kel.to_csv(OUTPUT_DIR / "kelurahan_label_correction_audit.csv", index=False, encoding="utf-8-sig")


## 6. Audit missing Kelurahan
Missing **tidak diimputasi**. Karena polanya tidak acak antar kecamatan, dampaknya dihitung per kecamatan.

In [ ]:
m_kel = df["kel_norm"].isna()
print("Missing Kelurahan:", int(m_kel.sum()), f"({m_kel.mean()*100:.2f}%)")
print("Missing Kelurahan tetapi Kecamatan ada:", int((m_kel & df["Kecamatan"].notna()).sum()))

dampak = (df.groupby("Kecamatan").agg(total=("raw_row", "size"), missing_kelurahan=("kel_norm", lambda s: int(s.isna().sum())))
          .assign(persen_missing=lambda t: (t["missing_kelurahan"] / t["total"] * 100).round(2))
          .sort_values("missing_kelurahan", ascending=False))
print("Dampak per Kecamatan (10 teratas):")
display(dampak.head(10))
dampak.to_csv(OUTPUT_DIR / "missing_kelurahan_impact_per_kecamatan.csv", encoding="utf-8-sig")
display(pd.crosstab(df.loc[m_kel, "Kecamatan"], df.loc[m_kel, "Tahun_file"], margins=True).sort_values("All", ascending=False).head(8))

## 7. Teks `Keluhan`: cleaning ringan, audit ekstrem, duplikat, data pribadi


In [ ]:
def clean_complaint_text(v):
    if pd.isna(v):
        return pd.NA
    v = str(v)
    
    v = re.sub(r"\\[nrt]", " ", v)
    v = unicodedata.normalize("NFKC", v)
    v = re.sub(r"\s+", " ", v).strip()
    return v if v else pd.NA

df["Keluhan_clean"] = df["Keluhan"].map(clean_complaint_text)
m_keluhan = df["Keluhan_clean"].isna()
print("Missing/kosong Keluhan:", int(m_keluhan.sum()))

_raw_txt = df["Keluhan"].astype("string")
n_artefak_raw = int(_raw_txt.str.contains(r"\\[nrt]", regex=True, na=False).sum())
n_artefak_sisa = int(df["Keluhan_clean"].astype("string").str.contains(r"\\[nrt]", regex=True, na=False).sum())
print(f"Baris dgn literal \\\\n/\\\\r/\\\\t di teks mentah : {n_artefak_raw}")
print(f"Sisa setelah cleaning (harus 0)           : {n_artefak_sisa}")
assert n_artefak_sisa == 0, "Masih ada literal backslash-escape tersisa setelah cleaning."

s = df["Keluhan_clean"].astype("string")
df["keluhan_char_count"] = s.str.len().fillna(0).astype(int)
df["keluhan_word_count"] = s.str.split().str.len().fillna(0).astype(int)
display(df.groupby("Tahun_file")[["keluhan_char_count", "keluhan_word_count"]].describe().T.round(1))


pendek = df[(df["keluhan_char_count"] > 0) & (df["keluhan_char_count"] <= SHORT_THRESHOLD)]
panjang = df[df["keluhan_char_count"] >= LONG_THRESHOLD]
print(f"\nTeks pendek (1..{SHORT_THRESHOLD} karakter): {len(pendek)} | teks panjang (>={LONG_THRESHOLD}): {len(panjang)}")
print("Isi teks pendek yang paling sering (apakah masih bermakna?):")
display(pendek["Keluhan_clean"].astype("string").str.lower().value_counts().head(12).rename("n").to_frame())
pd.concat([pendek.assign(extreme_type="VERY_SHORT_AUDIT"), panjang.assign(extreme_type="VERY_LONG_AUDIT")]) \
  [["No Ticket", "Tahun_file", "Kelurahan", "Keluhan_clean", "keluhan_char_count", "extreme_type"]] \
  .to_csv(OUTPUT_DIR / "text_extreme_audit.csv", index=False, encoding="utf-8-sig")

key_t = df["Keluhan_clean"].astype("string").str.casefold()
valid_key = key_t.notna() & df["kel_norm"].notna()
tmp = pd.DataFrame({"t": key_t, "k": df["kel_norm"].astype("string"), "y": df["tahun"]})
df["flag_dup_teks"] = valid_key & tmp.duplicated(keep=False)
print("\nBaris dengan teks identik di kelurahan & tahun yang sama (flag_dup_teks):", int(df["flag_dup_teks"].sum()))
rep = (tmp[df["flag_dup_teks"]].groupby(["t", "k", "y"]).size().reset_index(name="n").sort_values("n", ascending=False).head(8))
rep["t"] = rep["t"].str.slice(0, 80)
display(rep)

df["flag_telepon"] = s.fillna("").str.contains(PHONE_RE)
print("Baris memuat pola nomor telepon (flag_telepon):", int(df["flag_telepon"].sum()))


EMAIL_RE = re.compile(r"[\w.\-+]+@[\w\-]+(?:\.[\w\-]+)+")
ID_RE    = re.compile(r"(?<!\d)\d{15,}(?!\d)")            # NIK / No. KK / nomor identitas panjang lain
AKUN_RE  = re.compile(r"(?<![\w@])@\w{3,}")                 # @akun media sosial
URL_RE   = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)

def mask_telepon(v):
    if pd.isna(v):
        return pd.NA
    v = str(v)
    if MASK_PII_LAIN:                                         # urutan penting: url -> email -> identitas panjang -> telepon -> akun
        if MASK_URL:
            v = URL_RE.sub(TOKEN_URL, v)
        v = EMAIL_RE.sub(TOKEN_EMAIL, v)
        v = ID_RE.sub(TOKEN_ID, v)
    v = PHONE_RE.sub(TOKEN_TELEPON, v)
    if MASK_PII_LAIN:
        v = AKUN_RE.sub(TOKEN_AKUN, v)
    return v

df["Keluhan_embedding"] = df["Keluhan_clean"].map(mask_telepon) if MASK_TELEPON else df["Keluhan_clean"]
n_dimasking = int((df["Keluhan_embedding"].astype("string") != df["Keluhan_clean"].astype("string")).fillna(False).sum())
print(f"Baris yang teks_embedding-nya dimasking (MASK_TELEPON={MASK_TELEPON}, MASK_PII_LAIN={MASK_PII_LAIN}):", n_dimasking)
for _nm, _tok in [("telepon", TOKEN_TELEPON), ("identitas panjang", TOKEN_ID), ("email", TOKEN_EMAIL),
                  ("akun", TOKEN_AKUN), ("url", TOKEN_URL)]:
    print(f"  - memuat {_tok} ({_nm}):", int(df["Keluhan_embedding"].astype("string").str.contains(_tok, regex=False).fillna(False).sum()))

_emb = df["Keluhan_embedding"].astype("string").fillna("")
assert not _emb.str.contains(PHONE_RE).any(), "Masih ada pola telepon di teks_embedding"
assert not _emb.str.contains(EMAIL_RE).any(), "Masih ada email di teks_embedding"
AUDIT_RE = re.compile(r"(?:\d[\s.\-]?){9,}")                  # deret >= 9 digit: bisa nomor pelanggan/NOP/KTP
_aud = _emb[_emb.str.contains(AUDIT_RE)]
print(f"\nAudit longgar: {len(_aud)} baris masih memuat deret >= 9 digit (perlu ditinjau manual; bisa nomor pelanggan/objek pajak/dokumen).")
display(_aud.str.extract("(" + AUDIT_RE.pattern + ")")[0].str.replace(r"\d", "#", regex=True).str.strip()
        .value_counts().head(8).rename("n").to_frame())


## 7b. Masking nama (GLiNER)



In [ ]:
import pickle
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()           

MASK_NAMA = True
TOKEN_NAMA = "[NAMA]"
GLINER_MODEL = "urchade/gliner_multi-v2.1"   
LABELS_PII = ["person"]                       
THRESHOLD = 0.5
BATCH_SIZE = 16
CHECKPOINT_EVERY = 500
MAX_WORDS = 200          
OVERLAP_WORDS = 40
AUDIT_N, AUDIT_SEED = 300, 42

if MASK_NAMA:
    from gliner import GLiNER
    ner_model = GLiNER.from_pretrained(GLINER_MODEL)
    try:
        import torch
        if torch.cuda.is_available():
            ner_model = ner_model.to("cuda")
        elif torch.backends.mps.is_available():
            ner_model = ner_model.to("mps")   
        print("Perangkat:", next(ner_model.parameters()).device)
    except Exception as e:
        print("GPU tidak dipakai:", e)
    ner_model.eval()
    print("Model:", GLINER_MODEL, "| max_len:", ner_model.config.max_len)
else:
    print("MASK_NAMA=False: Keluhan_tanpanama akan sama dengan Keluhan_embedding.")

In [ ]:
def _gabung_token(t):
    return re.sub(re.escape(TOKEN_NAMA) + r"(?:\s*" + re.escape(TOKEN_NAMA) + r")+", TOKEN_NAMA, t)

def _word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]

def _merge_overlaps(ents):
    ents = sorted(ents, key=lambda e: (-float(e.get("score", 0)), -(e["end"] - e["start"])))
    kept = []
    for e in ents:
        if all(e["end"] <= k["start"] or e["start"] >= k["end"] for k in kept):
            kept.append(e)
    return sorted(kept, key=lambda e: e["start"])

def mask_nama(text, ents):
    for e in sorted(_merge_overlaps(ents), key=lambda e: e["start"], reverse=True):
        if str(e.get("label", "")).lower() != "person":
            continue
        if not (0 <= e["start"] < e["end"] <= len(text)):
            continue
        text = text[:e["start"]] + TOKEN_NAMA + text[e["end"]:]
    return _gabung_token(text)

def predict_entities_long(text):
    spans = _word_spans(text)
    ents, step = [], MAX_WORDS - OVERLAP_WORDS
    for s in range(0, len(spans), step):
        w = spans[s:s + MAX_WORDS]
        c0, c1 = w[0][0], w[-1][1]
        for e in ner_model.predict_entities(text[c0:c1], LABELS_PII, threshold=THRESHOLD):
            ents.append({**e, "start": e["start"] + c0, "end": e["end"] + c0})
        if s + MAX_WORDS >= len(spans):
            break
    return ents

def predict_batch(txts):
    if hasattr(ner_model, "batch_predict_entities"):
        return ner_model.batch_predict_entities(txts, LABELS_PII, threshold=THRESHOLD)
    return [ner_model.predict_entities(t, LABELS_PII, threshold=THRESHOLD) for t in txts]


In [ ]:
assert df["No Ticket"].astype(str).is_unique, "No Ticket harus unik sebagai kunci checkpoint."

if not MASK_NAMA:
    df["Keluhan_tanpanama"] = df["Keluhan_embedding"]
else:
    CHECKPOINT_PATH = OUTPUT_DIR / "masking_nama_checkpoint_PRIVAT.pkl"
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "rb") as f:
            hasil_masking = pickle.load(f)["hasil_masking"]
        print("Melanjutkan checkpoint:", len(hasil_masking), "tiket sudah diproses.")
    else:
        hasil_masking = {}

    src = pd.Series(df["Keluhan_embedding"].to_numpy(), index=df["No Ticket"].astype(str).to_numpy())
    kunci = list(src.index)
    n_potongan = 0

    for a in range(0, len(kunci), CHECKPOINT_EVERY):
        pending = [k for k in kunci[a:a + CHECKPOINT_EVERY] if k not in hasil_masking]
        if not pending:
            continue
        pendek, panjang = [], []
        for k in pending:
            t = src.loc[k]
            if pd.isna(t) or str(t).strip() == "":
                hasil_masking[k] = pd.NA
            elif len(_word_spans(str(t))) <= MAX_WORDS:
                pendek.append(k)
            else:
                panjang.append(k)

        for b in range(0, len(pendek), BATCH_SIZE):
            ks = pendek[b:b + BATCH_SIZE]
            txts = [str(src.loc[k]) for k in ks]
            for k, t, ents in zip(ks, txts, predict_batch(txts)):
                hasil_masking[k] = mask_nama(t, ents)

        for k in panjang:
            t = str(src.loc[k])
            hasil_masking[k] = mask_nama(t, predict_entities_long(t))
        n_potongan += len(panjang)

        with open(CHECKPOINT_PATH, "wb") as f:
            pickle.dump({"hasil_masking": hasil_masking}, f)
        print(f"Progres: {len(hasil_masking):,}/{len(kunci):,} tiket")

    df["Keluhan_tanpanama"] = df["No Ticket"].astype(str).map(hasil_masking).astype("string")
    print("Tiket yang diproses per potongan (teks panjang):", n_potongan)


In [ ]:

_emb = df["Keluhan_embedding"].astype("string")
_tnm = df["Keluhan_tanpanama"].astype("string")
assert _tnm.isna().equals(_emb.isna()), "Kekosongan teks berubah setelah masking."

n_nama_rows = int(_tnm.str.contains(TOKEN_NAMA, regex=False).fillna(False).sum())
n_tanpanama_diubah = int((_tnm != _emb).fillna(False).sum())
print(f"Tiket memuat {TOKEN_NAMA}: {n_nama_rows} dari {len(df)} (semua baris raw)")
print(f"Tiket yang berubah setelah masking nama: {n_tanpanama_diubah}")

rasio = (_tnm.str.len() / _emb.str.len()).dropna()
print(f"Rasio panjang teks setelah/sebelum: median={rasio.median():.3f}, min={rasio.min():.3f}")
if rasio.min() < 0.3:
    print("PERINGATAN: ada teks yang menyusut > 70%. Periksa apakah label person terlalu agresif.")

smp = _tnm.dropna().sample(AUDIT_N, random_state=AUDIT_SEED)
audit = pd.DataFrame({"sample_id": np.arange(1, len(smp) + 1),
                      "teks_masked": smp.str.slice(0, 600).to_numpy(),
                      "sisa_nama": "", "salah_masking": ""})
AUDIT_PATH = OUTPUT_DIR / "audit_masking_nama_PRIVAT.csv"
audit.to_csv(AUDIT_PATH, index=False, encoding="utf-8-sig")
print("Sampel audit disimpan di:", AUDIT_PATH)


## 7c. Hasil audit manual masking nama (jalankan setelah kolom `sisa_nama` dan `salah_masking` diisi 0/1)

Laju kebocoran = proporsi tiket sampel yang masih memuat nama orang setelah masking; laju salah masking = proporsi tiket yang memuat `[NAMA]` pada kata yang bukan nama orang.


In [ ]:
def _wilson(k, n, z=1.96):
    p = k / n
    lo = (p + z*z/(2*n) - z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))) / (1 + z*z/n)
    hi = (p + z*z/(2*n) + z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))) / (1 + z*z/n)
    return p, lo, hi

if not AUDIT_PATH.exists():
    print("Berkas audit belum ada; jalankan sel 7b-4 dulu.")
else:
    a = pd.read_csv(AUDIT_PATH)
    isi = a[["sisa_nama", "salah_masking"]].apply(pd.to_numeric, errors="coerce")
    if isi.isna().any().any():
        print("Kolom sisa_nama / salah_masking belum terisi penuh (0/1). Isi dulu di:", AUDIT_PATH)
    else:
        n = len(a)
        for kol, ket in [("sisa_nama", "masih memuat nama orang"), ("salah_masking", "salah masking")]:
            k = int((isi[kol] == 1).sum()); p, lo, hi = _wilson(k, n)
            print(f"Tiket {ket}: {k}/{n} = {p:.3f} (95% Wilson: {lo:.3f} - {hi:.3f})")


## 8. Kelompok topik (16 kelompok)

In [ ]:
def kelompok_topik(topik, instansi):
    teks = str(topik).lower() + " " + str(instansi).lower()
    if any(k in teks for k in ["pju", "lampu jalan", "penerangan jalan", "traffic light", "lampu penerangan"]):
        return "Penerangan Jalan & Traffic Light"
    if any(k in teks for k in ["jalan rusak", "jalan berlubang", "pengaspalan", "trotoar", "pedestrian", "jalan", "saluran tepi jalan"]):
        return "Infrastruktur Jalan & Saluran"
    if any(k in teks for k in ["banjir", "genangan", "drainase"]):
        return "Banjir & Drainase"
    if any(k in teks for k in ["sampah", "pemangkasan", "perantingan", "pohon", "kebersihan", "lingkungan"]):
        return "Kebersihan & Lingkungan"
    if any(k in teks for k in ["parkir", "rambu", "zebra cross", "bollard", "angkutan", "dishub", "lalu lintas", "perhubungan"]):
        return "Perhubungan & Lalu Lintas"
    if any(k in teks for k in ["kesehatan", "puskesmas", "bpjs", "rumah sakit", "catin"]):
        return "Kesehatan"
    if any(k in teks for k in ["ktp", "kk", "akta", "kependudukan", "dukcapil", "catatan sipil"]):
        return "Kependudukan & Catatan Sipil"
    if any(k in teks for k in ["pendidikan", "sekolah", "ijazah", "spp", "pip", "kip"]):
        return "Pendidikan"
    if any(k in teks for k in ["keamanan", "ketertiban", "preman", "balap liar", "satpol pp", "satpol"]):
        return "Keamanan & Ketertiban"
    if any(k in teks for k in ["pbb", "bphtb", "pajak", "retribusi"]):
        return "Pajak & Retribusi Daerah"
    if any(k in teks for k in ["perumahan", "pertanahan", "pbg", "skrk", "rusunawa", "rumah susun"]):
        return "Perumahan & Pertanahan"
    if any(k in teks for k in ["sosial", "kemiskinan", "miskin", "mbr", "bansos"]):
        return "Sosial"
    if any(k in teks for k in ["tenaga kerja", "lowongan", "pemagangan", "disnaker"]):
        return "Ketenagakerjaan"
    if any(k in teks for k in ["pdam", "air bersih", "kabel", "utilitas", "listrik"]):
        return "Utilitas"
    if any(k in teks for k in ["kecamatan", "kelurahan", "kewilayahan", "pelayanan kelurahan"]):
        return "Pelayanan Kewilayahan"
    return "Lainnya"

KELOMPOK_VALID = {
    "Penerangan Jalan & Traffic Light", "Infrastruktur Jalan & Saluran", "Banjir & Drainase",
    "Kebersihan & Lingkungan", "Perhubungan & Lalu Lintas", "Kesehatan", "Kependudukan & Catatan Sipil",
    "Pendidikan", "Keamanan & Ketertiban", "Pajak & Retribusi Daerah", "Perumahan & Pertanahan",
    "Sosial", "Ketenagakerjaan", "Utilitas", "Pelayanan Kewilayahan", "Lainnya",
}
assert set(TOPIK_OVERRIDE.values()) <= KELOMPOK_VALID, "TOPIK_OVERRIDE memuat nama kelompok yang tidak dikenal."

def _kelompok_final(t, i):
    if (t, i) in TOPIK_OVERRIDE:
        return TOPIK_OVERRIDE[(t, i)]
    if t in TOPIK_OVERRIDE:
        return TOPIK_OVERRIDE[t]
    return kelompok_topik(t, i)

_topik_ada = set(df["Topik"].dropna()); _pair_ada = set(zip(df["Topik"], df["Instansi"]))
_tak_terpakai = [k for k in TOPIK_OVERRIDE if (k not in _pair_ada if isinstance(k, tuple) else k not in _topik_ada)]
assert not _tak_terpakai, f"TOPIK_OVERRIDE memuat kunci yang tidak ada di data (salah ketik?): {_tak_terpakai}"

kombinasi = df[["Topik", "Instansi"]].drop_duplicates().copy()
kombinasi["kelompok_topik"] = [
    _kelompok_final(t, i) for t, i in zip(kombinasi["Topik"], kombinasi["Instansi"])
]
df = df.drop(columns=["kelompok_topik"], errors="ignore").merge(kombinasi, on=["Topik", "Instansi"], how="left")
assert df["kelompok_topik"].notna().all() and set(df["kelompok_topik"]) <= KELOMPOK_VALID

dist = (df["kelompok_topik"].value_counts() / len(df) * 100).round(1).rename("persen").to_frame()
display(dist)

print("\nAudit: topik yang jatuh ke 'Lainnya' (kandidat salah kelompok):")
display(df.loc[df["kelompok_topik"] == "Lainnya", "Topik"].value_counts().head(12).rename("n").to_frame())
print("Audit: topik terbanyak di 'Pelayanan Kewilayahan' (kelompok penampung; periksa apakah ada topik infrastruktur/drainase):")
display(df.loc[df["kelompok_topik"] == "Pelayanan Kewilayahan", "Topik"].value_counts().head(12).rename("n").to_frame())

n_komb = df.groupby(["Topik", "Instansi"], dropna=False).size().rename("n_tiket").reset_index()
komb_n = kombinasi.merge(n_komb, on=["Topik", "Instansi"], how="left")

sampel = komb_n.sample(frac=1, random_state=42).groupby("kelompok_topik").head(5).sort_values(["kelompok_topik", "n_tiket"], ascending=[True, False])
sampel.to_csv(OUTPUT_DIR / "topik_sample_audit.csv", index=False, encoding="utf-8-sig")
print("\nSampel kombinasi Topik+Instansi per kelompok disimpan ke topik_sample_audit.csv (periksa apakah kelompoknya tepat).")

KATA_LUAS = ["jalan", "kk", "pip", "kip", "spp", "mbr", "pju"]
teks_k = komb_n["Topik"].astype("string").fillna("").str.lower() + " " + komb_n["Instansi"].astype("string").fillna("").str.lower()
luas = pd.concat([komb_n[teks_k.str.contains(k, regex=False)].assign(kata_luas=k) for k in KATA_LUAS], ignore_index=True)
luas = luas.sort_values("n_tiket", ascending=False)
luas.to_csv(OUTPUT_DIR / "topik_kata_luas_audit.csv", index=False, encoding="utf-8-sig")
print("Audit: kombinasi yang mengandung kata kunci luas (mis. 'pip' juga menangkap 'pipa'; 'jalan' juga 'perjalanan'):")
display(luas.head(25))

komb_n["kelompok_aturan"] = [kelompok_topik(t, i) for t, i in zip(komb_n["Topik"], komb_n["Instansi"])]
efek = (komb_n[komb_n["kelompok_aturan"] != komb_n["kelompok_topik"]]
        .groupby(["Topik", "kelompok_aturan", "kelompok_topik"])["n_tiket"].sum().reset_index()
        .sort_values("n_tiket", ascending=False))
efek.to_csv(OUTPUT_DIR / "topik_override_effect.csv", index=False, encoding="utf-8-sig")
print(f"\nEfek TOPIK_OVERRIDE: {int(efek['n_tiket'].sum())} tiket berpindah kelompok")
display(efek)


## 9. Keputusan inklusi


In [ ]:
m_tgl = df["tanggal_dt"].isna()
alasan = pd.DataFrame({
    "MISSING_KELURAHAN": df["kel_norm"].isna(),
    "MISSING_KELUHAN": df["Keluhan_clean"].isna(),
    "INVALID_DATE": m_tgl,
})
assert int(alasan["INVALID_DATE"].sum()) == n_tgl_invalid, "Jumlah tanggal invalid di bagian 3 tidak sama dengan yang dikeluarkan."
df["exclusion_reason"] = alasan.apply(lambda r: ";".join(r.index[r]), axis=1)
df["include_in_clean"] = df["exclusion_reason"] == ""

df_clean = df[df["include_in_clean"]].copy()
excluded = df[~df["include_in_clean"]].copy()

print("Raw:", len(df), "| Clean:", len(df_clean), "| Excluded:", len(excluded))
display(df["exclusion_reason"].replace("", "(dipertahankan)").value_counts().rename("n").to_frame())
excluded[["raw_row", "No Ticket", "Tahun_file", "Kelurahan", "Kecamatan", "Topik", "Keluhan", "exclusion_reason"]] \
    .to_csv(OUTPUT_DIR / "excluded_rows.csv", index=False, encoding="utf-8-sig")

n_multi = int(excluded["exclusion_reason"].str.contains(";", regex=False).sum())
print(f"Baris unik dikeluarkan: {len(excluded)} | di antaranya punya >1 alasan: {n_multi}")
print("Kemunculan tiap alasan (bisa tumpang tindih, total boleh melebihi jumlah baris unik):")
display(alasan.sum().astype(int).rename("n_baris_dengan_alasan").to_frame())


## 10. Validasi (hard checks)
Semua pemeriksaan harus **PASS** sebelum file clean ditulis.

In [ ]:
per_th = df_clean.groupby("tahun")["lokasi_key"].nunique()
per_id_kec = df_clean.groupby("lokasi_key")["kecamatan_master"].nunique()
per_id_lat = df_clean.groupby("lokasi_key")[["lat", "lon"]].nunique()
per_id_lbl = df_clean.groupby("lokasi_key")["kelurahan_master"].nunique()
pair_ke_id = df_clean.groupby(["kelurahan_master", "kecamatan_master"])["lokasi_key"].nunique()
ID_MASTER  = set(master["lokasi_key"])
wakil_thn  = df_clean.groupby("tahun")["lokasi_key"].agg(lambda s: set(s) == ID_MASTER)

checks = {
    "raw = clean + excluded": len(df) == len(df_clean) + len(excluded),
    "tiket unik pada data clean": df_clean["No Ticket"].is_unique,
    "tidak ada Kelurahan kosong": df_clean["kel_norm"].notna().all(),
    "tidak ada Keluhan kosong": df_clean["Keluhan_clean"].notna().all(),
    "tidak ada tanggal invalid": df_clean["tanggal_dt"].notna().all(),
    "tahun (tanggal) == tahun (file)": (df_clean["tahun"] == df_clean["Tahun_file"]).all(),
    "semua baris clean MATCH_COORDINATE": (df_clean["coordinate_status"] == "MATCH_COORDINATE").all(),
    "tidak ada lokasi AMBIGU pada data clean": (df_clean["metode_cocok"] != "AMBIGU").all(),
    "koreksi kecamatan hanya lewat identitas tak ambigu": df_clean.loc[df_clean["kecamatan_dikoreksi"], "metode_cocok"].isin(["NAMA_UNIK", "LABEL_MASTER", "MANUAL_TERVERIFIKASI"]).all(),
    "koreksi label kelurahan hanya untuk nama kembar & kecamatan mentah = kecamatan master": bool(
        df_clean.loc[df_clean["kelurahan_dikoreksi"], "kec_k_raw"].astype(str).eq(df_clean.loc[df_clean["kelurahan_dikoreksi"], "kec_k_master"].astype(str)).all()
        and df_clean.loc[df_clean["kelurahan_dikoreksi"], "lokasi_key"].str.split("|").str[0].isin(KELUARGA_NAMA.keys()).all()),
    "tanpa koordinat fallback": ALLOW_FALLBACK_COORDINATE or (~df_clean["sumber_koordinat"].astype(str).str.contains("fallback", case=False)).all(),
    f"tepat {N_KELURAHAN} identitas (kelurahan+kecamatan) pada setiap tahun": bool((per_th == N_KELURAHAN).all() and len(per_th) == len(TAHUN_LIST)),
    f"seluruh {N_KELURAHAN} identitas administratif master terwakili pada setiap tahun": bool(wakil_thn.all() and len(wakil_thn) == len(TAHUN_LIST)),
    "tiap identitas tepat 1 kecamatan": bool((per_id_kec == 1).all()),
    "tiap identitas tepat 1 koordinat": bool((per_id_lat == 1).all().all()),
    "tiap identitas tepat 1 label kelurahan": bool((per_id_lbl == 1).all()),
    "tiap pasangan (kelurahan, kecamatan) tepat 1 identitas": bool((pair_ke_id == 1).all()),
    "kelompok_topik valid": set(df_clean["kelompok_topik"]) <= KELOMPOK_VALID,
}
hasil = pd.DataFrame({"pemeriksaan": list(checks), "status": ["PASS" if v else "FAIL" for v in checks.values()]})
display(hasil)
gagal = hasil[hasil["status"] == "FAIL"]["pemeriksaan"].tolist()
assert not gagal, f"Validasi GAGAL: {gagal}"
print("SEMUA VALIDASI PASS.")


## 11. Audit coverage

In [ ]:
ringkas_tahun = (df.groupby("Tahun_file").agg(raw=("raw_row", "size"))
                 .join(df_clean.groupby("Tahun_file").agg(clean=("raw_row", "size"))).fillna(0).astype(int))
ringkas_tahun["excluded"] = ringkas_tahun["raw"] - ringkas_tahun["clean"]
ringkas_tahun["retention_pct"] = (ringkas_tahun["clean"] / ringkas_tahun["raw"] * 100).round(2)
display(ringkas_tahun)

per_kel_tahun = df_clean.groupby(["tahun", "lokasi_key", "kelurahan_master", "kecamatan_master"]).size().rename("n_pengaduan").reset_index()
print("Distribusi jumlah pengaduan per kelurahan-tahun:")
display(per_kel_tahun.groupby("tahun")["n_pengaduan"].describe().round(1))
sedikit = per_kel_tahun[per_kel_tahun["n_pengaduan"] < MIN_N_WARN]
print(f"Kelurahan-tahun dengan < {MIN_N_WARN} pengaduan (embedding rata-rata akan noisy):", len(sedikit))
display(sedikit.head(20))
sedikit.to_csv(OUTPUT_DIR / "kelurahan_tahun_pengaduan_sedikit.csv", index=False)

## 12. Export

In [ ]:
TEKS_MASKED = df_clean["Keluhan_tanpanama"] if MASK_NAMA else df_clean["Keluhan_embedding"]

export = pd.DataFrame({
    "no_tiket": df_clean["No Ticket"],
    "tahun_data": df_clean["Tahun_file"],
    "tahun": df_clean["tahun"].astype(int),
    "tanggal_keluhan": df_clean["tanggal_dt"],
    "instansi": df_clean["Instansi"],
    "topik": df_clean["Topik"],
    "kelompok_topik": df_clean["kelompok_topik"],
    "kategori": df_clean["Kategori"],
    "channel": df_clean["Channel"],
    "teks_embedding": TEKS_MASKED,          
    "kelurahan": df_clean["kelurahan_master"],
    "kecamatan": df_clean["kecamatan_master"],
    "lokasi_key": df_clean["lokasi_key"],
    "metode_cocok": df_clean["metode_cocok"],
    "lat_kelurahan": df_clean["lat"],
    "lon_kelurahan": df_clean["lon"],
    "sumber_koordinat": df_clean["sumber_koordinat"],
    "kelurahan_raw": df_clean["Kelurahan"],
    "kecamatan_raw": df_clean["Kecamatan"],
    "kecamatan_dikoreksi": df_clean["kecamatan_dikoreksi"],
    "kelurahan_dikoreksi": df_clean["kelurahan_dikoreksi"],
    "keluhan_char_count": df_clean["keluhan_char_count"],
    "keluhan_word_count": df_clean["keluhan_word_count"],
    "flag_dup_teks": df_clean["flag_dup_teks"],
    "flag_telepon": df_clean["flag_telepon"],
    "raw_row": df_clean["raw_row"],
}).reset_index(drop=True)
if "kode_kel" in df_clean.columns:
    export.insert(export.columns.get_loc("kecamatan") + 1, "kode_kel", df_clean["kode_kel"].to_numpy())

assert "teks_keluhan" not in export.columns, "Kolom teks mentah tidak boleh ikut disimpan."
assert len(export) == len(df_clean)
if MASK_NAMA:
    assert export["teks_embedding"].notna().sum() == df_clean["Keluhan_embedding"].notna().sum(), "Ada teks yang hilang saat masking nama."
    print("Tiket berisi [NAMA]:", int(export["teks_embedding"].astype("string").str.contains(TOKEN_NAMA, regex=False).fillna(False).sum()))

export.to_csv(OUTPUT_DIR / "data_cleaned.csv", index=False, encoding="utf-8-sig")

for f in ["data_cleaned_tanpanama.csv"]:
    if (OUTPUT_DIR / f).exists():
        (OUTPUT_DIR / f).unlink(); print("Dihapus:", OUTPUT_DIR / f)

summary = pd.DataFrame([
    ("raw_rows", len(df)), ("clean_rows", len(df_clean)), ("excluded_rows", len(excluded)),
    ("excluded_missing_kelurahan_kemunculan", int(alasan["MISSING_KELURAHAN"].sum())),
    ("excluded_missing_keluhan_kemunculan", int(alasan["MISSING_KELUHAN"].sum())),
    ("excluded_invalid_date_kemunculan", int(alasan["INVALID_DATE"].sum())),
    ("excluded_rows_multi_alasan", n_multi),
    ("kecamatan_dikoreksi_rows", int(df_clean["kecamatan_dikoreksi"].sum())),
    ("kelurahan_dikoreksi_rows", int(df_clean["kelurahan_dikoreksi"].sum())),
    ("teks_artefak_backslash_n_dibersihkan", n_artefak_raw),
    ("unique_lokasi_clean", int(df_clean["lokasi_key"].nunique())),
    ("unique_kecamatan_clean", int(df_clean["kecamatan_master"].nunique())),
    ("flag_dup_teks_rows", int(df_clean["flag_dup_teks"].sum())),
    ("flag_telepon_rows", int(df_clean["flag_telepon"].sum())),
    ("kelurahan_tahun_lt_min_n", len(sedikit)),
    ("teks_embedding_dimasking_rows", int((df_clean["Keluhan_embedding"].astype("string") != df_clean["Keluhan_clean"].astype("string")).fillna(False).sum())),   # pada data CLEAN (sama dengan CSV)
    ("teks_embedding_dimasking_rows_raw_sebelum_eksklusi", n_dimasking),                                                                                         # pada seluruh raw (termasuk baris yang dikeluarkan)
    ("teks_tanpanama_mengandung_NAMA_rows", int(df_clean["Keluhan_tanpanama"].astype("string").str.contains(TOKEN_NAMA, regex=False).fillna(False).sum())),
    ("teks_tanpanama_diubah_rows", int((df_clean["Keluhan_tanpanama"].astype("string") != df_clean["Keluhan_embedding"].astype("string")).fillna(False).sum())),
] + [(f"cocok_{k}", int(v)) for k, v in df_clean["metode_cocok"].value_counts().items()] + [
], columns=["metric", "value"])
summary.to_csv(OUTPUT_DIR / "cleaning_summary.csv", index=False)
(OUTPUT_DIR / "cleaning_config.json").write_text(json.dumps({
    "files": {str(k): str(v) for k, v in FILES.items()}, "coord_file": str(COORD_FILE),
    "kelurahan_alias": KELURAHAN_ALIAS, "topik_override": {k if isinstance(k, str) else " | ".join(map(str, k)): v for k, v in TOPIK_OVERRIDE.items()},
    "lokasi_manual": {" | ".join(k): v for k, v in LOKASI_MANUAL.items()}, "mask_telepon": MASK_TELEPON,
    "allow_fallback_coordinate": ALLOW_FALLBACK_COORDINATE,
    "mask_nama": MASK_NAMA, "gliner_model": GLINER_MODEL, "gliner_threshold": THRESHOLD,
}, indent=2, ensure_ascii=False), encoding="utf-8")

display(summary)
print("Tersimpan di:", OUTPUT_DIR / "data_cleaned.csv")